# 24 — Graph RAG (Knowledge Graph + Multi-hop Retrieval)

Build a directed knowledge graph from documents, then answer questions by traversing entity relationships — capturing multi-hop facts that similarity search misses entirely.

**What you'll learn**
- Why similarity search fails on relational questions ("who founded the company that makes X?")
- Extracting `(subject, predicate, object)` triples with `with_structured_output`
- Building a `networkx.DiGraph` from triples — nodes are entities, edges are relationships
- Traversing the graph: find entities in the question, collect neighbors, pass as LLM context
- Three-node graph: `find_entities → retrieve_subgraph → generate`

**Companion examples:** 22-hybrid-search-rag (BM25+vector for single-doc facts), 13-structured-output (triple extraction pattern)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai networkx python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Build the knowledge graph from documents ───────────────────────────────
# For each document, extract (subject, predicate, object) triples with an LLM.
# Add each triple as a directed edge in a NetworkX graph.
#
# This is the graph-build phase: N LLM calls (one per doc).
# Once built, retrieval is pure graph traversal — no more LLM calls until generate.
import networkx as nx
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

DOCS = [
    "LangChain was founded by Harrison Chase in 2022 as an open-source Python framework for LLM applications.",
    "LangGraph is a library built on top of LangChain by Harrison Chase, adding stateful graph-based agent workflows.",
    "LangSmith is an observability platform created by LangChain Inc. for tracing and evaluating LLM pipelines.",
    "Anthropic was founded by Dario Amodei and Daniela Amodei in 2021 after they left OpenAI.",
    "Claude is the AI assistant developed by Anthropic, designed with a focus on safety and helpfulness.",
    "OpenAI was co-founded by Sam Altman, Elon Musk, Greg Brockman, and Ilya Sutskever in 2015.",
    "GPT-4 is a large language model developed by OpenAI, released in March 2023.",
    "Ilya Sutskever co-founded OpenAI and later founded Safe Superintelligence Inc. (SSI) in 2024.",
    "Dario Amodei previously served as VP of Research at OpenAI before co-founding Anthropic.",
    "LangChain Inc. raised a $25M Series A in 2023 and later a $35M round led by Sequoia Capital.",
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class Triple(BaseModel):
    subject: str
    predicate: str
    object: str


class Triples(BaseModel):
    triples: list[Triple]


def extract_triples(doc: str) -> list[Triple]:
    extractor = llm.with_structured_output(Triples)
    return extractor.invoke(
        [
            SystemMessage(
                "Extract factual (subject, predicate, object) triples. "
                "Use short canonical names (e.g. 'LangChain', not 'the LangChain framework'). "
                "Return only triples clearly stated in the text."
            ),
            HumanMessage(doc),
        ]
    ).triples


G = nx.DiGraph()
print("Building knowledge graph (1 LLM call per document)...")
for doc in DOCS:
    for t in extract_triples(doc):
        G.add_edge(t.subject, t.object, predicate=t.predicate)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("Sample nodes:", list(G.nodes())[:8])

In [ ]:
# ── 4. Inspect the multi-hop paths ────────────────────────────────────────────
# The question "Who founded the company that makes LangGraph?" requires:
#   LangGraph -> (built on) -> LangChain -> (founded by) -> Harrison Chase
# A single-document retrieval can find the LangGraph doc OR the LangChain doc,
# but not both in context together. The graph naturally surfaces both.


def get_neighbors(entity: str) -> list[str]:
    facts = []
    for _, n, data in G.out_edges(entity, data=True):
        facts.append(f"{entity} {data['predicate']} {n}")
    for src, _, data in G.in_edges(entity, data=True):
        facts.append(f"{src} {data['predicate']} {entity}")
    return facts


print("Facts connected to 'LangGraph':")
for f in get_neighbors("LangGraph"):
    print(f"  {f}")

print("\nFacts connected to 'LangChain' (one hop further):")
for f in get_neighbors("LangChain"):
    print(f"  {f}")

In [ ]:
# ── 5. Build the three-node graph and run queries ─────────────────────────────
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

TOP_K = 3


class GraphRAGState(TypedDict):
    question: str
    entities: list  # entity names extracted from the question
    context: str  # subgraph facts assembled for the LLM
    answer: str


class EntityMatch(BaseModel):
    entities: list[str]


def find_entities(state: GraphRAGState) -> dict:
    matcher = llm.with_structured_output(EntityMatch)
    result = matcher.invoke(
        [
            SystemMessage(
                f"Extract key named entities from this question. "
                f"Match to names in this list where possible: {list(G.nodes())}. "
                "Return canonical entity names only."
            ),
            HumanMessage(state["question"]),
        ]
    )
    return {"entities": result.entities}


def retrieve_subgraph(state: GraphRAGState) -> dict:
    facts, seen = [], set()
    for entity in state["entities"]:
        matches = [
            n for n in G.nodes() if entity.lower() in n.lower() or n.lower() in entity.lower()
        ]
        for node in matches[:2]:
            for f in get_neighbors(node):
                if f not in seen:
                    facts.append(f)
                    seen.add(f)
    context = "\n".join(facts[: TOP_K * 4]) if facts else "No relevant graph context found."
    return {"context": context}


def generate(state: GraphRAGState) -> dict:
    response = llm.invoke(
        [
            SystemMessage(
                "Answer the question using only the knowledge graph facts below. "
                "If the answer cannot be derived from these facts, say so.\n\n"
                f"Graph facts:\n{state['context']}"
            ),
            HumanMessage(state["question"]),
        ]
    )
    return {"answer": response.content}


graph = StateGraph(GraphRAGState)
graph.add_node("find_entities", find_entities)
graph.add_node("retrieve_subgraph", retrieve_subgraph)
graph.add_node("generate", generate)
graph.add_edge(START, "find_entities")
graph.add_edge("find_entities", "retrieve_subgraph")
graph.add_edge("retrieve_subgraph", "generate")
graph.add_edge("generate", END)
app = graph.compile()

QUESTIONS = [
    "Who founded the company that makes LangGraph?",  # multi-hop
    "What did Ilya Sutskever do after leaving OpenAI?",  # direct
    "What company did Dario Amodei leave to found Anthropic?",  # multi-hop
]

for q in QUESTIONS:
    result = app.invoke({"question": q, "entities": [], "context": "", "answer": ""})
    print(f"Q: {q}")
    print(f"Entities matched: {result['entities']}")
    print(f"A: {result['answer']}\n")

## Exercises

**Exercise 1 — Add a document:** Append `"Mistral AI was founded by Arthur Mensch in 2023."` to DOCS. Re-build the graph. Ask `"Who founded Mistral?"` — does the graph answer it?

**Exercise 2 — Visualize the graph:** Run:
```python
import matplotlib.pyplot as plt
nx.draw(G, with_labels=True, node_size=500, font_size=8)
plt.show()
```
Can you trace the multi-hop paths visually?

**Exercise 3 — Compare with vector RAG:** Build a `Chroma` vectorstore from the same `DOCS`. Ask the same multi-hop questions. Does vector search retrieve the right documents? What context does it pass to the LLM?

## What's next

- **22-hybrid-search-rag** — BM25+vector for factual single-doc retrieval
- **13-structured-output** — triple extraction pattern in depth with Pydantic schemas
- **25-adaptive-rag** — classify queries first, then route to the right retrieval strategy